# 🌐 Scraping GitHub Issues — Diskusi Pemrograman .NET
### File output: `so_dotnet_dataset_scraped.csv`

Data diambil langsung dari **GitHub REST API** (tanpa token, gratis).  
File CSV yang dihasilkan notebook ini **wajib digunakan** sebagai input notebook pelatihan.

| Item | Detail |
|---|---|
| Sumber | GitHub REST API v3 |
| Topik | .net, aspnetcore, ef-core, blazor, wpf, maui, signalr, linq, dll |
| Output CSV | `so_dotnet_dataset_scraped.csv` |


## 📦 Cell 1 — Install & Import

In [ ]:
!pip install requests pandas tqdm openpyxl matplotlib --quiet

import requests, time, re, os, warnings
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

# ── Nama file output TETAP — digunakan juga di notebook pelatihan ──
CSV_OUTPUT  = "so_dotnet_dataset_scraped.csv"   # ← nama wajib sama dengan notebook pelatihan
OUTPUT_DIR  = "/content"

print("✅ Library siap!")
print(f"📄 File output akan disimpan sebagai : {CSV_OUTPUT}")


## ⚙️ Cell 2 — Konfigurasi Scraping

In [ ]:
GITHUB_TOKEN = ""       # Opsional — isi untuk 5000 req/jam (tanpa token: 10 req/menit)
BASE_URL     = "https://api.github.com/search/issues"
DELAY        = 8        # detik antar request (aman untuk tanpa token)

HEADERS = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"token {GITHUB_TOKEN}"

# Query topik .NET — (query_string, label_topik, maks_halaman)
QUERIES = [
    ("dotnet+bug+language:csharp",          "bug",           3),
    ("aspnetcore+language:csharp",          "aspnetcore",    3),
    ("entityframework+language:csharp",     "ef-core",       2),
    ("blazor+language:csharp",              "blazor",        2),
    ("dotnet+performance+language:csharp",  "performance",   2),
    ("csharp+wpf+issue",                    "wpf",           2),
    ("dotnet+feature+language:csharp",      "feature",       2),
    ("maui+dotnet+language:csharp",         "maui",          2),
    ("signalr+language:csharp",             "signalr",       2),
    ("csharp+linq+issue",                   "linq",          2),
    ("dotnet+security+language:csharp",     "security",      1),
    ("csharp+dependency+injection",         "di",            2),
    ("dotnet+async+csharp",                 "async",         2),
    ("csharp+serialization+json",           "serialization", 2),
    ("dotnet+testing+xunit",                "testing",       2),
]

print(f"Total query  : {len(QUERIES)}")
print(f"Delay        : {DELAY}s per request")
print(f"Output file  : {CSV_OUTPUT}")


## 🔧 Cell 3 — Fungsi Scraping

In [ ]:
def scrape_github_issues(queries, delay=DELAY):
    """
    Scraping GitHub Issues via REST API.
    Mengembalikan DataFrame dengan kolom standar untuk pelatihan ML.
    """
    records  = []
    seen_ids = set()

    pbar = tqdm(queries, desc="Scraping topik")
    for query, topik, max_pages in pbar:
        for page in range(1, max_pages + 1):
            pbar.set_postfix(topik=topik, page=page, total=len(records))
            try:
                r = requests.get(
                    f"{BASE_URL}?q={query}&per_page=100&page={page}&sort=updated&order=desc",
                    headers=HEADERS, timeout=15)

                remaining = r.headers.get("X-RateLimit-Remaining", "?")

                if r.status_code == 403:
                    tqdm.write("⛔ Rate limit — tunggu 65 detik...")
                    time.sleep(65)
                    continue
                if r.status_code != 200:
                    tqdm.write(f"  ⚠️  HTTP {r.status_code} — skip")
                    break

                items = r.json().get("items", [])

                for item in items:
                    iid = item.get("id")
                    if iid in seen_ids or not item.get("title"):
                        continue
                    seen_ids.add(iid)

                    # Bersihkan body HTML/Markdown
                    body = str(item.get("body") or "")
                    body = re.sub(r"```[\s\S]*?```", " CODE_BLOCK ", body)
                    body = re.sub(r"`[^`]+`",          " INLINE_CODE ", body)
                    body = re.sub(r"!\[.*?\]\(.*?\)","", body)
                    body = re.sub(r"\[.*?\]\(.*?\)", " LINK ", body)
                    body = re.sub(r"#+\s",             " ", body)
                    body = re.sub(r"\s+",              " ", body).strip()

                    lbl_list  = [l["name"].lower() for l in item.get("labels", [])]
                    react_obj = item.get("reactions", {})
                    total_react = react_obj.get("total_count", 0) if isinstance(react_obj, dict) else 0

                    records.append({
                        "id":             iid,
                        "judul":          item.get("title", "").strip(),
                        "topik":          topik,
                        "labels_github":  ", ".join(lbl_list),
                        "state":          item.get("state", "open"),
                        "komentar":       item.get("comments", 0),
                        "reactions":      total_react,
                        "body_clean":     body[:500],
                        "panjang_judul":  len(item.get("title", "")),
                        "panjang_body":   len(body),
                        "ada_code_block": int("CODE_BLOCK" in body),
                        "tanggal_dibuat": str(item.get("created_at", ""))[:10],
                        "ditutup":        1 if item.get("closed_at") else 0,
                        "url":            item.get("html_url", ""),
                        "author":         (item.get("user") or {}).get("login", ""),
                    })

                tqdm.write(
                    f"  [{topik}] hal {page}: +{len([x for x in records])} total "
                    f"| quota={remaining}"
                )

                if len(items) < 100:
                    break
                time.sleep(delay)

            except Exception as e:
                tqdm.write(f"  ❌ Error: {e}")
                break

    df = pd.DataFrame(records)
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
    return df

print("✅ Fungsi scraping siap.")


## 🚀 Cell 4 — Jalankan Scraping

In [ ]:
print(f"Mulai scraping: {datetime.now().strftime('%H:%M:%S')}")
print(f"Output CSV    : so_dotnet_dataset_scraped.csv\n")

df_scraped = scrape_github_issues(QUERIES, delay=DELAY)

print(f"\n✅ Scraping selesai!")
print(f"   Total item : {len(df_scraped):,}")
print(f"   Kolom      : {list(df_scraped.columns)}")
print(f"\nDistribusi topik:")
print(df_scraped["topik"].value_counts().to_string())


## 💾 Cell 5 — Simpan Dataset ke `so_dotnet_dataset_scraped.csv`

> File ini yang **wajib digunakan** sebagai input notebook pelatihan.


In [ ]:
import os

# ── Simpan CSV dengan nama tetap ─────────────────────────────
CSV_SAVE_PATH = os.path.join(OUTPUT_DIR, CSV_OUTPUT)

df_scraped.index += 1
df_scraped.index.name = "no"
df_scraped.to_csv(CSV_SAVE_PATH, index=True, encoding="utf-8-sig")

file_size_kb = os.path.getsize(CSV_SAVE_PATH) / 1024

print("=" * 55)
print("  FILE CSV HASIL SCRAPING TERSIMPAN")
print("=" * 55)
print(f"  Nama file  : so_dotnet_dataset_scraped.csv")
print(f"  Path       : {CSV_SAVE_PATH}")
print(f"  Ukuran     : {file_size_kb:,.0f} KB")
print(f"  Total baris: {len(df_scraped):,}")
print(f"  Kolom      : {len(df_scraped.columns)}")
print()
print("⚠️  PENTING: File ini digunakan sebagai INPUT notebook pelatihan")
print(f"   Nama file harus tetap: so_dotnet_dataset_scraped.csv")


## 📊 Cell 6 — Eksplorasi Data (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"EDA Dataset Hasil Scraping — {len(df_scraped):,} item",
             fontweight="bold")

tv = df_scraped["topik"].value_counts()
axes[0].barh(tv.index, tv.values, color="steelblue", edgecolor="white")
axes[0].set_title("Distribusi Topik Hasil Scraping")
axes[0].set_xlabel("Jumlah Issue"); axes[0].invert_yaxis()
for i, v in enumerate(tv.values):
    axes[0].text(v+1, i, str(v), va="center", fontsize=9)

sv = df_scraped["state"].value_counts()
axes[1].pie(sv.values, labels=sv.index, autopct="%1.1f%%",
            colors=["#2ecc71","#e74c3c"],
            startangle=90, wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Status Issue (open vs closed)")

plt.tight_layout()
plt.savefig("/content/eda_scraping.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total issue open  : {(df_scraped['state']=='open').sum():,}")
print(f"Total issue closed: {(df_scraped['state']=='closed').sum():,}")
print(f"Avg komentar      : {df_scraped['komentar'].mean():.1f}")


## 🔎 Cell 7 — Preview & Download CSV

In [ ]:
print("5 baris pertama CSV hasil scraping:")
display(df_scraped[["id","judul","topik","state","komentar","ditutup","url"]].head())

print(f"\nInfo dataset:")
print(f"  File      : so_dotnet_dataset_scraped.csv")
print(f"  Baris     : {len(df_scraped):,}")
print(f"  Kolom     : {list(df_scraped.columns)}")

# Download CSV ke komputer
from google.colab import files
print("\n⬇️  Download CSV hasil scraping...")
files.download(CSV_SAVE_PATH)
print(f"✅ File {CSV_SAVE_PATH} berhasil didownload")
print(f"   → Gunakan file ini sebagai input notebook pelatihan")
